# 05 — Multi-horizon machine-learning models

This notebook trains and evaluates the machine-learning family used to forecast greenhouse air temperature and relative humidity at five temporal resolutions and four forecast horizons.

The workflow follows two validation-only selection phases:

1. **Phase A — model and hyperparameter selection:** Random Forest, Extra Trees, Histogram Gradient Boosting, and XGBoost are compared using two parameter candidates and the `SHT_TIME` feature set.
2. **Phase B — feature-set ablation:** the selected algorithm at each temporal resolution is evaluated with `SHT_BASE`, `SHT_TIME`, `SHT_AUX_TIME`, and `SHT_MULTISENSOR_TIME`.

The independent test partition is never used for model, hyperparameter, or feature-set selection. Every configuration within a temporal resolution uses the effective forecast origins produced by notebooks 03 and 04.

**Inputs**

- Resolution datasets in `data/processed/resolutions/`
- Effective forecast-origin indices in `data/processed/effective_indices/`

**Main outputs**

- Predictions and metrics in `results/machine_learning/`
- Trained models in `models/machine_learning/`
- Summary figure in PNG and PDF format in `figures/machine_learning/`

All paths are relative to the repository root.


In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message=r"Skipping features without any observed values.*",
    category=UserWarning,
    module=r"sklearn\.impute\._base",
)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if not XGBOOST_AVAILABLE:
    raise ImportError(
        "XGBoost is required for the four-algorithm comparison. "
        "Install the project requirements and restart the kernel."
    )


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "resolutions").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run notebook 03 first and keep the standard folder structure."
    )


PROJECT_ROOT = find_project_root()
RESOLUTION_DIR = PROJECT_ROOT / "data" / "processed" / "resolutions"
INDEX_DIR = PROJECT_ROOT / "data" / "processed" / "effective_indices"
RESULTS_DIR = PROJECT_ROOT / "results" / "machine_learning"
MODEL_DIR = PROJECT_ROOT / "models" / "machine_learning"
FIGURE_DIR = PROJECT_ROOT / "figures" / "machine_learning"

for directory in [RESULTS_DIR, MODEL_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Resolution datasets: {RESOLUTION_DIR.relative_to(PROJECT_ROOT)}")
print(f"Effective indices: {INDEX_DIR.relative_to(PROJECT_ROOT)}")


## Experimental parameters

Each input vector summarizes the preceding 24 hours. Dynamic variables are represented by lags, changes from the current value, rolling statistics, and the 24-hour missing-value fraction. Calendar covariates are evaluated at the forecast origin.

The BME280 relative-humidity series is not interpolated after saturation filtering. It appears only in `SHT_MULTISENSOR_TIME`; missing feature values are imputed with training-partition medians inside the model pipeline. `SHT_AUX_TIME` uses BME280 temperature and pressure, but not BME280 relative humidity. No auxiliary-data rule changes the common forecast origins.


In [ ]:
RESOLUTIONS = [4, 12, 20, 30, 60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS_MINUTES = [60, 120, 240, 480]
HISTORY_HOURS = 24

# The 24 h and 48 h lag candidates are retained in the declared candidate grid.
# Only lags strictly shorter than the 24 h history and windows no longer than it are active.
LAG_HOURS = [0, 1, 2, 3, 6, 12, 24, 48]
ROLLING_HOURS = [1, 3, 6, 12, 24, 48]
ROLLING_STATISTICS = ["mean", "std", "min", "max"]
MIN_ROLLING_COVERAGE = 0.50
ACTIVE_LAG_HOURS = [hour for hour in LAG_HOURS if hour == 0 or hour < HISTORY_HOURS]
ACTIVE_ROLLING_HOURS = [hour for hour in ROLLING_HOURS if hour <= HISTORY_HOURS]

STATIC_FEATURES = ["hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos"]

FEATURE_SETS = {
    "SHT_BASE": {
        "history": ["temperature", "relative_humidity"],
        "static": [],
    },
    "SHT_TIME": {
        "history": ["temperature", "relative_humidity"],
        "static": STATIC_FEATURES,
    },
    "SHT_AUX_TIME": {
        "history": ["temperature", "relative_humidity", "temperature_bme280", "pressure"],
        "static": STATIC_FEATURES,
    },
    "SHT_MULTISENSOR_TIME": {
        "history": [
            "temperature", "relative_humidity", "temperature_bme280",
            "relative_humidity_bme280", "pressure", "rh_bme280_saturated_fraction",
        ],
        "static": STATIC_FEATURES,
    },
}

PARAMETER_GRID = {
    "RANDOM_FOREST": [
        {"n_estimators": 140, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"},
        {"n_estimators": 200, "max_depth": 18, "min_samples_leaf": 2, "max_features": 0.7},
    ],
    "EXTRA_TREES": [
        {"n_estimators": 160, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"},
        {"n_estimators": 220, "max_depth": 18, "min_samples_leaf": 2, "max_features": 0.7},
    ],
    "HIST_GRADIENT_BOOSTING": [
        {"max_iter": 160, "learning_rate": 0.06, "max_leaf_nodes": 31, "l2_regularization": 0.0},
        {"max_iter": 220, "learning_rate": 0.04, "max_leaf_nodes": 63, "l2_regularization": 0.2},
    ],
    "XGBOOST": [
        {"n_estimators": 180, "learning_rate": 0.06, "max_depth": 5, "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 1.0},
        {"n_estimators": 260, "learning_rate": 0.04, "max_depth": 6, "subsample": 0.85, "colsample_bytree": 0.80, "reg_lambda": 2.0},
    ],
}


## Load the common forecast origins

The origin files are outputs of notebooks 03 and 04. Their required columns, index limits, duplicate keys, and chronological order are checked before the feature matrices are constructed.


In [ ]:
def load_inputs(resolution_minutes):
    data_path = RESOLUTION_DIR / f"greenhouse_{resolution_minutes}min.csv"
    index_path = INDEX_DIR / f"effective_indices_{resolution_minutes}min.csv"
    if not data_path.exists() or not index_path.exists():
        raise FileNotFoundError(
            f"Missing inputs for {resolution_minutes} min. Run notebooks 03 and 04 first."
        )
    data = pd.read_csv(data_path, parse_dates=["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)
    origins = pd.read_csv(index_path, parse_dates=["origin_timestamp"])
    origins = origins.sort_values("origin_index").reset_index(drop=True)
    return data, origins


datasets = {}
effective_origins = {}
sample_rows = []
for resolution in RESOLUTIONS:
    data, origins = load_inputs(resolution)
    datasets[resolution] = data
    effective_origins[resolution] = origins

    required_index_columns = {
        "origin_index",
        "origin_timestamp",
        "split",
        *{f"target_index_h{horizon}" for horizon in HORIZONS_MINUTES},
        *{f"target_timestamp_h{horizon}" for horizon in HORIZONS_MINUTES},
    }
    missing_index_columns = sorted(required_index_columns.difference(origins.columns))
    if missing_index_columns:
        raise KeyError(
            f"Missing index columns for {resolution} min: {missing_index_columns}"
        )

    indices_within_dataset = bool(
        origins["origin_index"].between(0, len(data) - 1).all()
        and all(
            origins[f"target_index_h{horizon}"].between(0, len(data) - 1).all()
            for horizon in HORIZONS_MINUTES
        )
    )
    duplicate_origin_indices = int(origins["origin_index"].duplicated().sum())
    duplicate_origin_timestamps = int(origins["origin_timestamp"].duplicated().sum())

    if not indices_within_dataset:
        raise IndexError(f"Forecast indices exceed the {resolution}-minute dataset.")
    if duplicate_origin_indices or duplicate_origin_timestamps:
        raise ValueError(f"Duplicate forecast origins found for {resolution} minutes.")

    split_counts = origins["split"].value_counts()
    sample_rows.append(
        {
            "resolution": f"{resolution}min",
            "dataset_rows": len(data),
            "total_origins": len(origins),
            "train": int(split_counts.get("train", 0)),
            "validation": int(split_counts.get("validation", 0)),
            "test": int(split_counts.get("test", 0)),
            "duplicate_origin_indices": duplicate_origin_indices,
            "duplicate_origin_timestamps": duplicate_origin_timestamps,
            "indices_within_dataset": indices_within_dataset,
        }
    )

sample_audit = pd.DataFrame(sample_rows)
sample_audit.to_csv(RESULTS_DIR / "01_input_sample_audit.csv", index=False)
display(sample_audit)


## Multi-horizon feature and target matrices

For a source variable, the notebook creates six lags, five changes, five rolling means, five rolling standard deviations, five rolling minima, five rolling maxima, and one missing-fraction feature. Therefore the feature-set widths are 64, 68, 132, and 196 variables respectively.


In [ ]:
def build_feature_matrix(data, origins, resolution_minutes, feature_set):
    specification = FEATURE_SETS[feature_set]
    origin_positions = origins["origin_index"].to_numpy(dtype=int)
    arrays, names, catalog = [], [], []

    for source in specification["history"]:
        series = data[source].astype(float)
        current = series.iloc[origin_positions].to_numpy(dtype=np.float32)

        for hours in ACTIVE_LAG_HOURS:
            steps = hours * 60 // resolution_minutes
            lagged = series.shift(steps).iloc[origin_positions].to_numpy(dtype=np.float32)
            lag_name = f"{source}__lag_{hours}h"
            arrays.append(lagged)
            names.append(lag_name)
            catalog.append({
                "resolution": f"{resolution_minutes}min", "feature_set": feature_set,
                "feature": lag_name, "source": source, "type": "lag", "hours": hours,
            })
            if hours > 0:
                delta_name = f"{source}__delta_{hours}h"
                arrays.append(current - lagged)
                names.append(delta_name)
                catalog.append({
                    "resolution": f"{resolution_minutes}min", "feature_set": feature_set,
                    "feature": delta_name, "source": source, "type": "difference", "hours": hours,
                })

        for hours in ACTIVE_ROLLING_HOURS:
            steps = hours * 60 // resolution_minutes
            min_periods = max(1, int(np.ceil(steps * MIN_ROLLING_COVERAGE)))
            rolling = series.rolling(steps, min_periods=min_periods)
            values_by_statistic = {
                "mean": rolling.mean(),
                "std": rolling.std(ddof=1),
                "min": rolling.min(),
                "max": rolling.max(),
            }
            for statistic in ROLLING_STATISTICS:
                feature_name = f"{source}__roll_{statistic}_{hours}h"
                arrays.append(
                    values_by_statistic[statistic].iloc[origin_positions].to_numpy(dtype=np.float32)
                )
                names.append(feature_name)
                catalog.append({
                    "resolution": f"{resolution_minutes}min", "feature_set": feature_set,
                    "feature": feature_name, "source": source,
                    "type": f"rolling_{statistic}", "hours": hours,
                })

        history_steps = HISTORY_HOURS * 60 // resolution_minutes
        missing_name = f"{source}__missing_fraction_{HISTORY_HOURS}h"
        missing_fraction = (
            series.isna().astype(float).rolling(history_steps, min_periods=1).mean()
            .iloc[origin_positions].to_numpy(dtype=np.float32)
        )
        arrays.append(missing_fraction)
        names.append(missing_name)
        catalog.append({
            "resolution": f"{resolution_minutes}min", "feature_set": feature_set,
            "feature": missing_name, "source": source,
            "type": "missing_fraction", "hours": HISTORY_HOURS,
        })

    for source in specification["static"]:
        feature_name = f"{source}__origin"
        arrays.append(data[source].iloc[origin_positions].to_numpy(dtype=np.float32))
        names.append(feature_name)
        catalog.append({
            "resolution": f"{resolution_minutes}min", "feature_set": feature_set,
            "feature": feature_name, "source": source, "type": "origin_static", "hours": 0,
        })

    matrix = pd.DataFrame(
        np.column_stack(arrays).astype(np.float32, copy=False), columns=names
    )
    return matrix, pd.DataFrame(catalog)


def build_target_matrix(data, origins):
    columns, values = [], []
    for target in TARGETS:
        for horizon in HORIZONS_MINUTES:
            columns.append(f"{target}__h{horizon}")
            positions = origins[f"target_index_h{horizon}"].astype(int).to_numpy()
            values.append(data.iloc[positions][target].to_numpy(dtype=np.float32))
    return pd.DataFrame(
        np.column_stack(values).astype(np.float32, copy=False), columns=columns
    )


feature_matrices = {}
target_matrices = {}
feature_catalog_frames = []
for resolution in RESOLUTIONS:
    data = datasets[resolution]
    origins = effective_origins[resolution]
    target_matrices[resolution] = build_target_matrix(data, origins)
    for feature_set in FEATURE_SETS:
        matrix, catalog = build_feature_matrix(data, origins, resolution, feature_set)
        feature_matrices[(resolution, feature_set)] = matrix
        feature_catalog_frames.append(catalog)

feature_catalog = (
    pd.concat(feature_catalog_frames, ignore_index=True)
    .drop_duplicates(["resolution", "feature_set", "feature"])
    .sort_values(["resolution", "feature_set", "feature"])
    .reset_index(drop=True)
)
feature_catalog.to_csv(RESULTS_DIR / "08_feature_catalog.csv", index=False)

feature_widths = pd.DataFrame([
    {"resolution": f"{resolution}min", "feature_set": feature_set, "n_features": matrix.shape[1]}
    for (resolution, feature_set), matrix in feature_matrices.items()
])
display(feature_widths.pivot(index="resolution", columns="feature_set", values="n_features"))
assert set(feature_widths.loc[feature_widths.feature_set == "SHT_BASE", "n_features"]) == {64}
assert set(feature_widths.loc[feature_widths.feature_set == "SHT_TIME", "n_features"]) == {68}
assert set(feature_widths.loc[feature_widths.feature_set == "SHT_AUX_TIME", "n_features"]) == {132}
assert set(feature_widths.loc[feature_widths.feature_set == "SHT_MULTISENSOR_TIME", "n_features"]) == {196}


## Model factories and evaluation utilities

Tree ensembles predict all eight target–horizon combinations jointly. Algorithms without native multi-output support are wrapped in `MultiOutputRegressor`. Every fitted estimator is preceded by training-only median imputation. Columns containing no observed values in a fitting subset are excluded by the imputer; the corresponding repetitive scikit-learn notification is suppressed while all other warnings and errors remain visible.


In [ ]:
def make_estimator(model_name, parameters):
    if model_name == "RANDOM_FOREST":
        estimator = RandomForestRegressor(
            **parameters, random_state=RANDOM_SEED, n_jobs=-1
        )
    elif model_name == "EXTRA_TREES":
        estimator = ExtraTreesRegressor(
            **parameters, random_state=RANDOM_SEED, n_jobs=-1
        )
    elif model_name == "HIST_GRADIENT_BOOSTING":
        base = HistGradientBoostingRegressor(
            **parameters, random_state=RANDOM_SEED, early_stopping=False
        )
        estimator = MultiOutputRegressor(base, n_jobs=-1)
    elif model_name == "XGBOOST":
        base = XGBRegressor(
            **parameters,
            objective="reg:squarederror",
            random_state=RANDOM_SEED,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
        estimator = MultiOutputRegressor(base, n_jobs=-1)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=False,
            ),
        ),
        ("model", estimator),
    ])


def split_mask(origins, split):
    return origins["split"].eq(split).to_numpy()


def task_metrics(y_true, y_pred, normalization_scale):
    rows = []
    for target_index, target in enumerate(TARGETS):
        for horizon_index, horizon in enumerate(HORIZONS_MINUTES):
            output_index = target_index * len(HORIZONS_MINUTES) + horizon_index
            observed = np.asarray(y_true[:, output_index], dtype=float)
            predicted = np.asarray(y_pred[:, output_index], dtype=float)
            valid = np.isfinite(observed) & np.isfinite(predicted)
            observed, predicted = observed[valid], predicted[valid]
            if not len(observed):
                raise ValueError(f"No finite values for {target}, horizon {horizon} min")
            rmse = float(np.sqrt(mean_squared_error(observed, predicted)))
            standard_deviation = float(normalization_scale[output_index])
            rows.append({
                "target": target,
                "horizon_minutes": horizon,
                "n": len(observed),
                "rmse": rmse,
                "nrmse": rmse / standard_deviation if standard_deviation > 1e-8 else np.nan,
                "mae": float(mean_absolute_error(observed, predicted)),
                "bias": float(np.mean(predicted - observed)),
                "r2": float(r2_score(observed, predicted)),
            })
    return pd.DataFrame(rows)


def aggregate_metrics(metrics):
    return {
        "mean_normalized_rmse": float(metrics["nrmse"].mean()),
        "mean_rmse": float(metrics["rmse"].mean()),
        "mean_r2": float(metrics["r2"].mean()),
        "minimum_r2": float(metrics["r2"].min()),
    }


def prediction_rows(origins, y_true, y_pred, resolution, model_name, feature_set, candidate_id, split):
    selected = origins.loc[origins["split"].eq(split)].reset_index(drop=True)
    rows = []
    for target_index, target in enumerate(TARGETS):
        for horizon_index, horizon in enumerate(HORIZONS_MINUTES):
            output_index = target_index * len(HORIZONS_MINUTES) + horizon_index
            observed = np.asarray(y_true[:, output_index], dtype=np.float32)
            predicted = np.asarray(y_pred[:, output_index], dtype=np.float32)
            frame = pd.DataFrame({
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "model": model_name,
                "feature_set": feature_set,
                "candidate_id": candidate_id,
                "split": split,
                "origin_index": selected["origin_index"].to_numpy(),
                "origin_timestamp": selected["origin_timestamp"].to_numpy(),
                "target_timestamp": selected[f"target_timestamp_h{horizon}"].to_numpy(),
                "target": target,
                "horizon_minutes": horizon,
                "observed": observed,
                "predicted": predicted,
            })
            frame["error"] = frame["predicted"] - frame["observed"]
            frame["absolute_error"] = frame["error"].abs()
            rows.append(frame)
    return pd.concat(rows, ignore_index=True)


## Phase A — model and hyperparameter selection

Each candidate is fitted on `train` and scored on `validation`. Within an algorithm, the candidate with the lowest mean validation NRMSE is retained; mean validation $R^2$ breaks ties. The winning algorithm for each temporal resolution is chosen by the same rule.


In [ ]:
tuning_rows = []
phase_a_metric_frames = []
phase_a_prediction_frames = []
selected_candidate_models = {}
available_models = [
    model for model in PARAMETER_GRID
    if model != "XGBOOST" or XGBOOST_AVAILABLE
]

for resolution in RESOLUTIONS:
    origins = effective_origins[resolution]
    X = feature_matrices[(resolution, "SHT_TIME")].to_numpy(dtype=float)
    Y = target_matrices[resolution].to_numpy(dtype=float)
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    test = split_mask(origins, "test")

    for model_name in available_models:
        candidate_records = []
        for candidate_id, parameters in enumerate(PARAMETER_GRID[model_name], start=1):
            estimator = make_estimator(model_name, parameters)
            start = time.perf_counter()
            estimator.fit(X[train], Y[train])
            fit_seconds = time.perf_counter() - start
            start = time.perf_counter()
            validation_prediction = estimator.predict(X[validation])
            inference_seconds = time.perf_counter() - start
            metrics = task_metrics(
                Y[validation], validation_prediction, np.std(Y[train], axis=0, ddof=1)
            )
            aggregate = aggregate_metrics(metrics)
            record = {
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "model": model_name,
                "feature_set": "SHT_TIME",
                "candidate_id": candidate_id,
                "parameters": json.dumps(parameters, sort_keys=True),
                "fit_seconds": fit_seconds,
                "inference_seconds": inference_seconds,
                **aggregate,
                "estimator": estimator,
                "validation_prediction": validation_prediction,
                "validation_metrics": metrics,
            }
            candidate_records.append(record)
            tuning_rows.append({key: value for key, value in record.items() if key not in {
                "estimator", "validation_prediction", "validation_metrics"
            }})

        winner = min(candidate_records, key=lambda row: row["mean_normalized_rmse"])
        selected_candidate_models[(resolution, model_name)] = winner

        validation_metrics = winner["validation_metrics"].copy()
        validation_metrics = validation_metrics.assign(
            resolution=f"{resolution}min",
            resolution_minutes=resolution,
            model=model_name,
            feature_set="SHT_TIME",
            candidate_id=winner["candidate_id"],
            split="validation",
            fit_seconds=winner["fit_seconds"],
            inference_seconds=winner["inference_seconds"],
        )
        phase_a_metric_frames.append(validation_metrics)
        phase_a_prediction_frames.append(prediction_rows(
            origins, Y[validation], winner["validation_prediction"], resolution,
            model_name, "SHT_TIME", winner["candidate_id"], "validation"
        ))

        final_estimator = make_estimator(model_name, json.loads(winner["parameters"]))
        fit_scope = train | validation
        start = time.perf_counter()
        final_estimator.fit(X[fit_scope], Y[fit_scope])
        test_fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        test_prediction = final_estimator.predict(X[test])
        test_inference_seconds = time.perf_counter() - start
        test_metrics = task_metrics(
            Y[test], test_prediction, np.std(Y[fit_scope], axis=0, ddof=1)
        ).assign(
            resolution=f"{resolution}min",
            resolution_minutes=resolution,
            model=model_name,
            feature_set="SHT_TIME",
            candidate_id=winner["candidate_id"],
            split="test",
            fit_seconds=test_fit_seconds,
            inference_seconds=test_inference_seconds,
        )
        phase_a_metric_frames.append(test_metrics)
        phase_a_prediction_frames.append(prediction_rows(
            origins, Y[test], test_prediction, resolution,
            model_name, "SHT_TIME", winner["candidate_id"], "test"
        ))

tuning_results = pd.DataFrame(tuning_rows)
tuning_results.to_csv(RESULTS_DIR / "03_hyperparameter_tuning.csv", index=False)
phase_a_metrics = pd.concat(phase_a_metric_frames, ignore_index=True)
phase_a_predictions = pd.concat(phase_a_prediction_frames, ignore_index=True)
phase_a_metrics.to_csv(RESULTS_DIR / "02_phase_a_metrics.csv", index=False)

selected_model_rows = []
for resolution in RESOLUTIONS:
    records = [selected_candidate_models[(resolution, model)] for model in available_models]
    winner = sorted(records, key=lambda row: (row["mean_normalized_rmse"], -row["mean_r2"]))[0]
    selected_model_rows.append({
        "resolution": f"{resolution}min",
        "resolution_minutes": resolution,
        "model": winner["model"],
        "feature_set": "SHT_TIME",
        "candidate_id": winner["candidate_id"],
        "parameters": winner["parameters"],
        "mean_normalized_rmse": winner["mean_normalized_rmse"],
        "mean_rmse": winner["mean_rmse"],
        "mean_r2": winner["mean_r2"],
        "minimum_r2": winner["minimum_r2"],
        "selection_rule": "lowest mean validation NRMSE; mean validation R2 as secondary criterion",
    })

selected_models = pd.DataFrame(selected_model_rows)
selected_models.to_csv(RESULTS_DIR / "04_selected_model_by_resolution.csv", index=False)
display(selected_models)


## Phase B — feature-set ablation

Only the algorithm and hyperparameters selected in Phase A are carried forward. Feature-set selection uses the lowest mean validation RMSE, with mean validation $R^2$ as the secondary criterion. The `SHT_TIME` result is reused from Phase A; all other feature sets are fitted separately.


In [ ]:
ablation_metric_frames = [phase_a_metrics.loc[
    phase_a_metrics.apply(
        lambda row: any(
            row["resolution_minutes"] == selected["resolution_minutes"]
            and row["model"] == selected["model"]
            for selected in selected_model_rows
        ), axis=1
    )
].copy()]
ablation_prediction_frames = [phase_a_predictions.loc[
    phase_a_predictions.apply(
        lambda row: any(
            row["resolution_minutes"] == selected["resolution_minutes"]
            and row["model"] == selected["model"]
            for selected in selected_model_rows
        ), axis=1
    )
].copy()]

fitted_test_models = {}
for selected in selected_model_rows:
    resolution = selected["resolution_minutes"]
    model_name = selected["model"]
    candidate_id = selected["candidate_id"]
    parameters = json.loads(selected["parameters"])
    origins = effective_origins[resolution]
    Y = target_matrices[resolution].to_numpy(dtype=float)
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    test = split_mask(origins, "test")

    for feature_set in FEATURE_SETS:
        if feature_set == "SHT_TIME":
            continue
        X = feature_matrices[(resolution, feature_set)].to_numpy(dtype=float)
        estimator = make_estimator(model_name, parameters)
        start = time.perf_counter()
        estimator.fit(X[train], Y[train])
        validation_fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        validation_prediction = estimator.predict(X[validation])
        validation_inference_seconds = time.perf_counter() - start
        metrics = task_metrics(
            Y[validation], validation_prediction, np.std(Y[train], axis=0, ddof=1)
        ).assign(
            resolution=f"{resolution}min",
            resolution_minutes=resolution,
            model=model_name,
            feature_set=feature_set,
            candidate_id=candidate_id,
            split="validation",
            fit_seconds=validation_fit_seconds,
            inference_seconds=validation_inference_seconds,
        )
        ablation_metric_frames.append(metrics)
        ablation_prediction_frames.append(prediction_rows(
            origins, Y[validation], validation_prediction, resolution,
            model_name, feature_set, candidate_id, "validation"
        ))

        final_estimator = make_estimator(model_name, parameters)
        fit_scope = train | validation
        start = time.perf_counter()
        final_estimator.fit(X[fit_scope], Y[fit_scope])
        test_fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        test_prediction = final_estimator.predict(X[test])
        test_inference_seconds = time.perf_counter() - start
        metrics = task_metrics(
            Y[test], test_prediction, np.std(Y[fit_scope], axis=0, ddof=1)
        ).assign(
            resolution=f"{resolution}min",
            resolution_minutes=resolution,
            model=model_name,
            feature_set=feature_set,
            candidate_id=candidate_id,
            split="test",
            fit_seconds=test_fit_seconds,
            inference_seconds=test_inference_seconds,
        )
        ablation_metric_frames.append(metrics)
        ablation_prediction_frames.append(prediction_rows(
            origins, Y[test], test_prediction, resolution,
            model_name, feature_set, candidate_id, "test"
        ))
        fitted_test_models[(resolution, feature_set)] = final_estimator

ablation_metrics = pd.concat(ablation_metric_frames, ignore_index=True)
ablation_predictions = pd.concat(ablation_prediction_frames, ignore_index=True)

validation_ablation = (
    ablation_metrics.loc[ablation_metrics["split"] == "validation"]
    .groupby(["resolution", "resolution_minutes", "model", "feature_set", "candidate_id"], as_index=False)
    .agg(mean_validation_rmse=("rmse", "mean"), mean_validation_r2=("r2", "mean"))
)
validation_ablation.to_csv(RESULTS_DIR / "05_feature_ablation_validation.csv", index=False)

selected_feature_rows = []
for resolution in RESOLUTIONS:
    subset = validation_ablation.loc[validation_ablation["resolution_minutes"] == resolution]
    winner = subset.sort_values(
        ["mean_validation_rmse", "mean_validation_r2"], ascending=[True, False]
    ).iloc[0]
    selected_feature_rows.append({
        **winner.to_dict(),
        "selection_rule": "lowest mean validation RMSE; mean validation R2 as secondary criterion",
    })

selected_features = pd.DataFrame(selected_feature_rows)
selected_features.to_csv(RESULTS_DIR / "06_selected_feature_set_by_resolution.csv", index=False)
display(validation_ablation.sort_values(["resolution_minutes", "mean_validation_rmse"]))
display(selected_features)


## Final artifacts and independent test evaluation

For each temporal resolution, the selected algorithm–feature combination is refitted on `train + validation`, serialized, and reported on `test`. The complete prediction file also retains the non-selected configurations so later notebooks can audit selection and compare model families.


In [ ]:
all_metrics = pd.concat([phase_a_metrics, ablation_metrics], ignore_index=True)
all_metrics = all_metrics.drop_duplicates(
    ["resolution", "model", "feature_set", "candidate_id", "split", "target", "horizon_minutes"]
).reset_index(drop=True)
all_predictions = pd.concat([phase_a_predictions, ablation_predictions], ignore_index=True)
all_predictions = all_predictions.drop_duplicates(
    ["resolution", "model", "feature_set", "candidate_id", "split", "origin_index", "target", "horizon_minutes"]
).reset_index(drop=True)

all_metrics.to_csv(RESULTS_DIR / "07_ml_metrics.csv", index=False)
all_predictions.to_csv(RESULTS_DIR / "09_ml_predictions.csv", index=False)

final_model_rows = []
for selected in selected_feature_rows:
    resolution = int(selected["resolution_minutes"])
    model_name = selected["model"]
    feature_set = selected["feature_set"]
    candidate_id = int(selected["candidate_id"])
    parameters = PARAMETER_GRID[model_name][candidate_id - 1]
    origins = effective_origins[resolution]
    X = feature_matrices[(resolution, feature_set)].to_numpy(dtype=np.float32)
    Y = target_matrices[resolution].to_numpy(dtype=np.float32)
    fit_scope = split_mask(origins, "train") | split_mask(origins, "validation")

    final_estimator = make_estimator(model_name, parameters)
    final_estimator.fit(X[fit_scope], Y[fit_scope])
    model_path = MODEL_DIR / f"{resolution}min" / model_name / feature_set / "final_model.joblib"
    model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({
        "pipeline": final_estimator,
        "resolution_minutes": resolution,
        "model": model_name,
        "feature_set": feature_set,
        "feature_names": feature_matrices[(resolution, feature_set)].columns.tolist(),
        "output_names": target_matrices[resolution].columns.tolist(),
        "candidate_id": candidate_id,
        "parameters": parameters,
        "random_seed": RANDOM_SEED,
        "fit_scope": "train+validation",
    }, model_path)
    final_model_rows.append({
        "resolution": f"{resolution}min",
        "resolution_minutes": resolution,
        "model": model_name,
        "feature_set": feature_set,
        "candidate_id": candidate_id,
        "parameters": json.dumps(parameters, sort_keys=True),
        "model_path": str(model_path.relative_to(PROJECT_ROOT)),
    })

final_models = pd.DataFrame(final_model_rows)
final_models.to_csv(RESULTS_DIR / "13_final_models.csv", index=False)

selected_test_metrics = all_metrics.merge(
    selected_features[["resolution_minutes", "model", "feature_set", "candidate_id"]],
    on=["resolution_minutes", "model", "feature_set", "candidate_id"],
    how="inner",
).loc[lambda frame: frame["split"] == "test"].copy()
selected_test_metrics.to_csv(RESULTS_DIR / "10_selected_ml_test_metrics.csv", index=False)

selected_test_predictions = all_predictions.merge(
    selected_features[["resolution_minutes", "model", "feature_set", "candidate_id"]],
    on=["resolution_minutes", "model", "feature_set", "candidate_id"],
    how="inner",
).loc[lambda frame: frame["split"] == "test"].copy()
selected_test_predictions.to_csv(RESULTS_DIR / "14_selected_ml_test_predictions.csv", index=False)

best_test_configurations = (
    selected_test_metrics.groupby(
        ["resolution", "resolution_minutes", "model", "feature_set", "candidate_id"], as_index=False
    )
    .agg(mean_test_rmse=("rmse", "mean"), mean_test_nrmse=("nrmse", "mean"), mean_test_r2=("r2", "mean"))
    .merge(
        final_models,
        on=["resolution", "resolution_minutes", "model", "feature_set", "candidate_id"],
        how="left",
        suffixes=("", "_model"),
    )
)
best_test_configurations.to_csv(RESULTS_DIR / "11_best_test_configurations.csv", index=False)

computational_summary = (
    all_metrics.groupby(["model", "feature_set", "split"], as_index=False)
    .agg(
        mean_fit_seconds=("fit_seconds", "mean"),
        total_fit_seconds=("fit_seconds", "sum"),
        mean_inference_seconds=("inference_seconds", "mean"),
        total_inference_seconds=("inference_seconds", "sum"),
        n_metric_rows=("rmse", "size"),
    )
)
computational_summary.to_csv(RESULTS_DIR / "12_computational_summary.csv", index=False)
display(best_test_configurations)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.lineplot(
    data=validation_ablation,
    x="resolution_minutes", y="mean_validation_rmse",
    hue="feature_set", marker="o", ax=axes[0],
)
axes[0].set_title("Feature-set ablation on validation data")
axes[0].set_xlabel("Temporal resolution (min)")
axes[0].set_ylabel("Mean validation RMSE")

sns.lineplot(
    data=selected_test_metrics,
    x="horizon_minutes", y="rmse",
    hue="target", style="resolution", markers=True, dashes=False, ax=axes[1],
)
axes[1].set_title("Selected machine-learning configurations")
axes[1].set_xlabel("Forecast horizon (min)")
axes[1].set_ylabel("Test RMSE")
axes[1].legend(fontsize=7, ncol=2)

png_path = FIGURE_DIR / "05_machine_learning_summary.png"
pdf_path = FIGURE_DIR / "05_machine_learning_summary.pdf"
fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()
print(f"Saved: {png_path.relative_to(PROJECT_ROOT)}")
print(f"Saved: {pdf_path.relative_to(PROJECT_ROOT)}")


In [ ]:
output_summary = pd.DataFrame(
    {
        "artifact": [
            "selected algorithms",
            "selected feature sets",
            "selected test metric rows",
            "selected test prediction rows",
            "serialized final models",
            "summary figure files",
        ],
        "count": [
            len(selected_models),
            len(selected_features),
            len(selected_test_metrics),
            len(selected_test_predictions),
            len(final_models),
            int(png_path.exists()) + int(pdf_path.exists()),
        ],
    }
)

display(selected_models)
display(selected_features)
display(output_summary)

print(f"Results: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Models: {MODEL_DIR.relative_to(PROJECT_ROOT)}")
print(f"Figures: {FIGURE_DIR.relative_to(PROJECT_ROOT)}")
